# FE-01 — Raw Dataset Audit

**Stage**: 01_dataset_audit (FE-01)

**Purpose**: Inspect the raw primary dataset (UCI Online Retail) and produce the initial
audit numbers that feed `docs/data_dictionary/data_dictionary.md`,
`docs/data_dictionary/dataset_schema.csv`, and
`docs/data_dictionary/initial_data_profile.md`.

**Out of scope**:

- preprocessing, cleaning, imputation, outlier handling
- feature engineering, RFM, customer aggregation
- scaling, clustering, evaluation, profiling

This notebook is **read-only**: it never modifies the raw xlsx file.

**Reproducibility**: the audit logic lives in `src/customer_segmentation/data/audit.py`
and the loader in `src/customer_segmentation/data/loader.py`. The orchestrator
`scripts/run_fe01_audit.py` is the canonical entry point and writes the same CSVs/MDs
as this notebook displays.

**Run order**:

1. `python scripts/run_fe01_audit.py`           — writes CSVs/MDs under `reports/fe01/`
2. `python scripts/generate_fe01_docs.py`      — writes the canonical docs in `docs/data_dictionary/`
3. This notebook                              — exploration and review (numbers must match step 1)

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

from customer_segmentation.data.audit import (
    basic_overview,
    categorical_profile,
    date_profile,
    duplicate_profile,
    frequency_table,
    identifier_profile,
    looks_like_cancellation_prefix,
    missing_profile,
    numerical_profile,
)
from customer_segmentation.data.loader import (
    list_sheet_names,
    load_raw_transactions,
)

## 1. Load the primary dataset

In [ ]:
PRIMARY_PATH = REPO_ROOT / "data" / "raw" / "primary" / "Online Retail.xlsx"
BACKUP_PATH = REPO_ROOT / "data" / "raw" / "backup" / "online_retail_II.xlsx"

sheets = list_sheet_names(PRIMARY_PATH)
print("Sheets in primary file:", sheets)
df = load_raw_transactions(PRIMARY_PATH, sheet_name=sheets[0])
print("Loaded primary shape:", df.shape)
df.head(3)

## 2. Top-level overview

In [ ]:
ov = basic_overview(df)
print(f"Rows: {ov.n_rows:,}")
print(f"Columns: {ov.n_columns}")
print(f"Memory (deep): {ov.memory_bytes:,} bytes")
print()
print("Columns and dtypes:")
for c, dt in ov.dtypes.items():
    print(f"  - {c}: {dt}")

## 3. Missing-value profile

In [ ]:
missing = missing_profile(df)
missing

## 4. Numerical profile

In [ ]:
num = numerical_profile(df)
num

## 5. Categorical profile

In [ ]:
cat = categorical_profile(df, top_k=10)
cat

## 6. Country distribution

In [ ]:
if "Country" in df.columns:
    country_freq = frequency_table(df["Country"], top_k=15)
    _ = country_freq

## 7. Date coverage

In [ ]:
if "InvoiceDate" in df.columns:
    dp = date_profile(df, "InvoiceDate")
    print(f"Parsed:  {dp.parsed_count:,} / {dp.total:,}")
    print(f"Unparsed: {dp.unparsed_count:,}")
    print(f"Min:     {dp.min}")
    print(f"Max:     {dp.max}")
    print(f"Duplicate timestamps: {dp.duplicate_timestamps:,}")
    print()
    print("Note: a 'reference_date_mode' for RFM is left for the FE-02 stage.")

## 8. Identifier analysis

In [ ]:
for col in ["InvoiceNo", "StockCode", "CustomerID"]:
    if col in df.columns:
        p = identifier_profile(df, col)
        print(
            f"{col}: total={p.total:,}, missing={p.missing_count:,}, "
            f"unique={p.unique_count:,}, duplicate={p.duplicate_count:,}, "
            f"unique_rate={p.unique_rate:.4f}%, is_unique={p.is_unique}"
        )

scan = looks_like_cancellation_prefix(df["InvoiceNo"], prefix="C")
print()
print(
    f"InvoiceNo with 'C' prefix: {scan['n_matching']:,} "
    f"({scan['rate_percent']:.4f}% of non-null)"
)
print(f"Examples: {scan['examples']}")

## 9. Transaction (invoice) structure

Note: we stay at transaction-line level here. No aggregation to customer is performed in FE-01.

In [ ]:
dup_exact = duplicate_profile(df)
dup_inv_sc = duplicate_profile(df, subset=["InvoiceNo", "StockCode"])
dup_full = duplicate_profile(
    df,
    subset=["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate"],
)
print(
    f"Exact-row duplicates: {dup_exact.exact_duplicate_rows:,} "
    f"({dup_exact.exact_duplicate_row_rate:.4f}%)"
)
print(f"(InvoiceNo, StockCode)-key duplicates: {dup_inv_sc.key_duplicate_rows:,}")
print(f"(InvoiceNo, StockCode)-key unique combinations: {dup_inv_sc.key_unique_combinations:,}")
print(
    "(InvoiceNo, StockCode, Description, Quantity, InvoiceDate)-key duplicates: "
    f"{dup_full.key_duplicate_rows:,}"
)
print()
print(
    "Interpretation: multiple rows per InvoiceNo is the natural structure "
    "of invoice-line-level data. A given InvoiceNo containing multiple "
    "products is NOT a data error."
)

## 10. Backup dataset — schema comparison only

In [ ]:
if BACKUP_PATH.exists():
    backup_sheets = list_sheet_names(BACKUP_PATH)
    print("Backup sheets:", backup_sheets)
    for s in backup_sheets:
        bdf = load_raw_transactions(BACKUP_PATH, sheet_name=s)
        bov = basic_overview(bdf)
        print(f"\n--- {s} ---")
        print(f"Rows: {bov.n_rows:,}; Columns: {bov.n_columns}")
        print(f"Columns: {list(bov.column_names)}")
        if "InvoiceDate" in bdf.columns:
            bdp = date_profile(bdf, "InvoiceDate")
            print(f"Date range: {bdp.min} -> {bdp.max}")
else:
    print("Backup file not present at", BACKUP_PATH)

## 11. Summary table (matches `docs/data_dictionary/dataset_schema.csv`)

In [ ]:
schema_csv = REPO_ROOT / "reports" / "fe01" / "primary_schema.csv"
if schema_csv.exists():
    schema = pd.read_csv(schema_csv)
    _ = schema
else:
    print(
        "Schema CSV not found at", schema_csv,
        "— run `python scripts/run_fe01_audit.py` first.",
    )

## 12. What FE-01 deliberately does NOT do

- It does **not** drop rows (including `CustomerID == NaN`, negative `Quantity`,
  zero/negative `UnitPrice`, missing `Description`, or duplicate rows).
- It does **not** impute missing values.
- It does **not** rename columns.
- It does **not** change `InvoiceNo` from mixed-type to a single type.
- It does **not** aggregate to customer level.
- It does **not** fit any model.

All of the above are policy decisions owned by FE-02 / PR-XX and must be
captured as ADRs in `docs/decisions/` before being implemented.